In [ ]:
import requests
import pandas as pd
import apimoex
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import datetime as dt
import os
import sklearn
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')
from math import *
import math

In [ ]:
stocks = {
    'AFLT': 'AF',
    'ALRS': 'AL',
    'CHMF': 'CH',
    'FEES': 'FS',
    'GAZR': 'GZ',
    'GMKN': 'GK',
    'HYDR': 'HY',
    'LKOH': 'LK',
    'MGNT': 'MN',
    'MOEX': 'ME',
    'MTSI': 'MT',
    'NLMK': 'NM',
    'NOTK': 'NK',
    'ROSN': 'RN',
    'RTKM': 'RT',
    'SBPR': 'SP',
    'SBRF': 'SR',
    'SNGP': 'SG',
    'SNGR': 'SN',
    'TATN': 'TT',
    'TRNF': 'TN',
    'VTBR': 'VB',
    'MAGN': 'MG',
    'PLZL': 'PZ',
    'YNDF': 'YN',
    'AFKS': 'AK',
    'IRAO': 'IR',
    'POLY': 'PO',
    'PIKK': 'PI',
    'SPBE': 'SE',
    'RUAL': 'RL',
    'PHOR': 'PH',
    'DSKY': 'DY',
    'SMLT': 'SS',
    'MTLR': 'MC',
    'RSTI': 'RE',
    'SIBN': 'SO',
    'TCSI': 'TI',
    'FIVE': 'FV',
    'MAIL': 'ML',
    'OZON': 'OZ',
    'SPYF': 'SF',
    'NASD': 'NA',
    'POSI': 'PS',
    'STOX': 'SX',
    'HANG': 'HS',
    'DAX': 'DX',
    'NIKK': 'N2',
    'ISKJ': 'IS',
    'WUSH': 'WU',
    'MVID': 'MV',
    'CBOM': 'CM',
    'SGZH': 'SZ',
    'BELU': 'BE',
    'FLOT': 'FL',
    'BSPB': 'BS',
    'BANE': 'BN',
    'KMAZ': 'KM'
}

commodities = {   
    'BR': 'BR',
    'GOLD': 'GD',
    'GL': 'GL',
    'GLDRUBF': 'GLDRUBF',
    'PLD': 'PD',
    'PLT': 'PT',
    'SILV': 'SV',
    'SUGR': 'SA',
    'SLV': 'SL',
    'ALMN': 'AM',
    'CL': 'CL',
    'Co': 'Co',
    'GLD': 'GO',
    'Nl': 'Nl',
    'Zn': 'Zn',
    'NG': 'NG',
    'WH4': 'WH',
    'WHEAT': 'W4',
    'SUGAR': 'Su'}

indexies = {
    'MIX': 'MX',
    'MXI': 'MM',
    'RTS': 'RI',
    'RTSM': 'RM',
    'RVI': 'VI',
    'HOME': 'HO',
    'OGI': 'OG',
    'MMI': 'MA',
    'FNI': 'FN',
    'CNI': 'CS',
    'RGBI': 'RB',
    'IMOEXF': 'IMOEXF'}

curr = {
    'CNY': 'CR',
    'Eu': 'Eu',
    'Si': 'Si',
    'USDRUBF': 'USDRUBF',
    'EURRUBF': 'EURRUBF',
    'CNYRUBF': 'CNYRUBF',
    'TRY': 'TY',
    'HKD': 'HK',
    'AED': 'AE',
    'INR': 'I2',
    'KZT': 'KZ',
    'AMD': 'AR',
    'ED': 'ED',
    'AUDU': 'AU',
    'GBPU': 'GU',
    'UCAD': 'CA',
    'UCHF': 'CF',
    'UJPY': 'JP',
    'UTRY': 'TR',
    'UCNY': 'UC',
    'ECAD': 'EC',
    'EGBP': 'EG',
    'EJPY': 'EJ'
}

month = {
    1: ['F', '01-01', '01-31'],
    2: ['G', '02-01', '02-28'],
    3: ['H', '03-01', '03-31'],
    4: ['J', '04-01', '04-30'],
    5: ['K', '05-01', '05-31'],
    6: ['M', '06-01', '06-30'],
    7: ['N', '07-01', '07-31'],
    8: ['Q', '08-01', '08-31'],
    9: ['U', '09-01', '09-30'],
    10: ['V', '10-01', '10-31'],
    11: ['X', '11-01', '11-30'],
    12: ['Z', '12-01', '12-31']
}
dict = stocks|indexies|commodities|curr


def closest_quots_candels(ISIN_pos,year):
    path = r'data'
    df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')
    df_data = pd.DataFrame()
    df_out = pd.DataFrame()
    expireation_date = '202'+str(year)+'-01-01'
    if ISIN_pos in commodities.keys():
        month_list = [1,2,3,4,5,6,7,8,9,10,11,12]
    else:
        month_list = [3,6,9,12]
    for i in month_list:
        try:
            if dict[ISIN_pos] in ['USDRUBF','EURRUBF','CNYRUBF','GLDRUBF']:
                instument = dict[ISIN_pos]
            else:
                instument = dict[ISIN_pos]+month[i][0]+str(year) #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx
            with requests.Session() as session:
                data = apimoex.get_market_candles(session=session,security=instument, start = expireation_date, market="forts",engine="futures",interval=24)
                df_data = pd.DataFrame(data)
                df_data['begin'] = df_data['begin'].apply(lambda x: x[:10])
                expireation_date = df_data['begin'][len(df_data['begin'])-1]
            df_out = pd.concat([df_out, df_data], ignore_index=True)

        except KeyError:
            print(i)
    return(df_out)

def closest_quots_voluemes(ISIN_pos,year):
    path = r'data'
    df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')
    df_data = pd.DataFrame()
    df_out = pd.DataFrame()
    expireation_date = '202'+str(year)+'-01-01'
    if ISIN_pos in commodities.keys():
        month_list = [1,2,3,4,5,6,7,8,9,10,11,12]
    else:
        month_list = [3,6,9,12]
    for i in month_list:
        try:
            if dict[ISIN_pos] in ['USDRUBF','EURRUBF','CNYRUBF','GLDRUBF']:
                instument = dict[ISIN_pos]
            else:
                instument = dict[ISIN_pos]+month[i][0]+str(year) #From MOEX site https://www.moex.com/ru/derivatives/contracts.aspx
            with requests.Session() as session:
                data = apimoex.get_market_history(session=session,security=instument, start = expireation_date, market="forts",engine="futures")
                df_data = pd.DataFrame(data)
                expireation_date = df_data['TRADEDATE'][len(df_data['TRADEDATE'])-1]
                df_data.insert (loc= 0 , column='Instrument', value=instument)
            df_out = pd.concat([df_out, df_data], ignore_index=True)
            

        except KeyError:
            print(i)
    df_out.rename(columns = {'TRADEDATE':'begin'}, inplace = True )
    return(df_out)


In [ ]:
def pin_values(data,instrument):

    
    df_quotes = data[data['isin'] == instrument]
    df_quotes['chg'] = df_quotes['close'].diff()
    df_quotes = df_quotes.dropna()

    
    def std(list):
        element = 0 
        for item in list:
            element = element + float((item**2)/( len(list)))
        return math.sqrt(element)
    
    def phi(x):
        #'Cumulative distribution function for the standard normal distribution'
        return (1.0 + erf(x / sqrt(2.0))) / 2.0
    
    def calx(v_i,delta_p_i,sigma):
        x = v_i * phi(delta_p_i/sigma)
        return x
    
    def main():
        global df
        df = df_quotes
    
    if __name__ == '__main__':
        main()
    
    high = df["high"].tolist()
    low = df["low"].tolist()
    vol = df["volume"].tolist()
    chg =df["chg"].tolist()
    date = df["moment"].tolist()
    #time = df["time"].tolist()
    sigma = std(chg) #calculate standard deviation
    totalvol = sum(vol)
    
    dateset = tuple(set(date))
    dates = []
    vpin=[]
    for eachdate in date:
        dateslice = df[df['moment'] == eachdate]
        datevol = dateslice['volume'].sum()
        vbs = datevol
        
        '''list to store results'''
        delta_p_i = []
        v_i = []
        x=[]
        
        '''convert slice to list'''
        highslice = dateslice["high"].tolist()
        lowslice = dateslice["low"].tolist()
        volslice = dateslice["volume"].tolist()
        
        '''generate basket'''
        highbas=[]
        lowbas=[]
        volbas=[]
        
        '''start filling basket'''
        for i in range(len(dateslice.index)):
            highbas.append(highslice[i])
            lowbas.append(lowslice[i])
            volbas.append(volslice[i])
    
            
            '''pop last element if vol exceeds.'''
            if sum(volbas) > vbs:
                v_i.append(sum(volbas[0:-1]))  
                delta_p_i.append(abs(max(highbas)-min(lowbas)))
                highbas = highbas[-1:] 
                lowbas = lowbas[-1:]
                volbas = volbas[-1:]
        for i in range(len(v_i)):
            x.append(calx(v_i[i],delta_p_i[i],sigma))
        v_tau_b=sum(x)
        dateitem= abs(2*v_tau_b - vbs)
        #print(eachdate,dateitem/totalvol)
        vpin.append(dateitem/totalvol)
        dates.append(eachdate)
    
    #PLOT
    # x = dates
    # x.sort()
    # #print(len(x))
    # #print(len(vpin))
    
    # x_titles = np.array(x)
    # fig,ax = plt.subplots()
    
    # ax.plot(x_titles , vpin , color='b',label='VPIN+instrument')
    # plt.xticks(x_titles[::10],size = 8,rotation = 45)
    
    
    # fig.set_size_inches(10, 6)
    # fig.savefig(r'plots/PIN/'+instrument+'.png')
        
    return pd.DataFrame({
        "moment": dates,
        "pin": vpin
    })

In [ ]:
def pin_quotes(data,instrument):
    df_for_graph = pd.merge(data[data['isin']==instrument],pin_values(data,instrument),on = 'moment').dropna()
    
    #График
    x_titles = np.array(df_for_graph['moment'].to_list())
    y_values = np.array(df_for_graph['close'].to_list())
    
    fig,ax = plt.subplots()
    
    plt.ylabel('Стоимость контракта, руб.')
    
    ax.plot(x_titles , y_values , color='b',label='цена')
    plt.xticks(x_titles[::10],size = 8,rotation = 45)
    plt.legend(loc = 2)
    
    y_values1 = np.array(df_for_graph['pin'].to_list())
    
    ax2 = ax.twinx ()
    ax2.plot(x_titles, y_values1[:] , color='r', label='pin')
    
    plt.legend(loc = 4)
    plt.ylabel("pin")
    
    fig.set_size_inches(10, 6)
    fig.savefig(r'plots/Local_PIN/'+instrument+'.png')


In [ ]:
path = r'data/OUT'
data = pd.read_csv(path+'/Pos+quotes+volume.csv')
interest_population = list(data['isin'].unique())

In [ ]:
for name in interest_population:
    pin_quotes(data,name)

In [ ]:
def pin_quotes_pos(instrument):
    path = r'data'
    df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')
    
    df_for_graph = pd.merge(df_ext['isin'],pin_graph(instrument,3),closest_quots_candels(instrument,3),on = 'begin').dropna()
    
    #График
    x_titles = np.array(df_for_graph['begin'].to_list())
    y_values = np.array(df_for_graph['close'].to_list())
    
    fig,ax = plt.subplots()
    
    plt.ylabel('Стоимость контракта, руб.')
    
    ax.plot(x_titles , y_values , color='b',label='цена')
    plt.xticks(x_titles[::10],size = 8,rotation = 45)
    plt.legend(loc = 2)
    
    y_values1 = np.array(df_for_graph['pin'].to_list())
    
    ax2 = ax.twinx ()
    ax2.plot(x_titles, y_values1[:] , color='r', label='pin')
    
    plt.legend(loc = 4)
    plt.ylabel("Кол-во контрактов")
    
    fig.set_size_inches(10, 6)
    fig.savefig(r'plots/'+instrument+'.png')


In [ ]:
path = r'data'
df_ext = pd.read_excel(path+'/OUT/All_days_consolidated.xlsx')


In [ ]:
list(dict.keys())[19:]